In [16]:
import pickle, os
import requests as reqlib

import pandas as pd
import numpy as np
import cma

In [17]:
reference_path = "../../results/transit/reference.parquet"
routing_endpoint = "http://localhost:8054/router/transit"
# routing_endpoint = "http://localhost:8029/router/transit"

output_path = "../../results/transit/calibration.p"

# Objective is observation_based or distribution_based
objective = "observation_based"

In [18]:
#if "papermill" in locals():
#    survey_path = papermill.input["survey"]
#    spatial_path = papermill.input["spatial"]

#    routing_endpoint = papermill.params["routing_endpoint"]
#    selected_objective = "individual"

#    progress_path = papermill.output["progress"]
#    output_path = papermill.output["parameters"]

In [19]:
# Load reference data
df_reference = pd.read_parquet(reference_path)

# df_reference = df_reference.iloc[:1000] # For testing

In [20]:
# Identify modes and maximum transfers
modes = [c.replace("legs_", "") for c in df_reference.columns if c.startswith("legs_")]
maximum_transfers = df_reference["transfers"].max()

In [21]:
# Convert to requests
requests = []

for index, row in df_reference.iterrows():
    requests.append({
        "request_index": int(row["request_index"]),
        "origin_x": row["origin_x"],
        "origin_y": row["origin_y"],
        "destination_x": row["destination_x"],
        "destination_y": row["destination_y"],
        "departure_time_s": row["departure_time"]
    })

In [22]:
# Prepare querying the routing server
def query_endpoint(requests, utilities):
    response = reqlib.post(routing_endpoint, json = {
        "batch": requests,
        "utilities": utilities
    })

    assert response.status_code == 200

    df_response = { 
        "request_index": [],
        "transfers": []
    }

    for mode in modes:
        df_response["legs_{}".format(mode)] = []

    for row in response.json():
        df_response["request_index"].append(row["request_index"])
        df_response["transfers"].append(np.minimum(row["transfers"], maximum_transfers))
        
        for mode in modes:
            if mode in row["vehicle_legs_by_mode"]:
                df_response["legs_{}".format(mode)].append(row["vehicle_legs_by_mode"][mode])
            else:
                df_response["legs_{}".format(mode)].append(0)
    
    return pd.DataFrame(df_response)

In [23]:
# We need to be careful about the request/response size, so we send individual batches
maximum_batch_size = 4000

def query_endpoint_batched(requests, utilities):
    df_response = []
    batch_index = 0

    while batch_index * maximum_batch_size < len(requests):
        df_response.append(query_endpoint(
            requests[batch_index * maximum_batch_size : (batch_index + 1) * maximum_batch_size],
            utilities))
        
        batch_index += 1
    
    return pd.concat(df_response)

In [24]:
# Define calibration variables
variables = [
    { "name": "rail_u_h", "initial": -1.0 },
    { "name": "subway_u_h", "initial": -1.0, "fixed": True },
    { "name": "bus_u_h", "initial": -1.0 },
    { "name": "tram_u_h", "initial": -1.0 },
    { "name": "other_u_h", "copy": "bus_u_h" },
    { "name": "wait_u_h", "initial": -1.0 },
    { "name": "walk_u_h", "initial": -1.0 },
    { "name": "transfer_u", "initial": -1.0 }
]

In [25]:
# Extend with index information for CMA-ES evaluation
variables_map = { v["name"]: v for v in variables }

active_index = 0

# First find active variables
for variable in variables:
    if "fixed" in variable or "copy" in variable: 
        continue

    variables_map[variable["name"]] = variable
    
    variable["index"] = active_index
    variable["optimized"] = True
    active_index += 1

# Then treat fixed variables and those copying from others
for variable in variables:
    if "fixed" in variable:
        variable["index"] = None
    
    if "copy" in variable:
        assert not "initial" in variable
        variable["initial"] = variables_map[variable["copy"]]["initial"]
        variable["index"] = variables_map[variable["copy"]]["index"]

In [26]:
# Define the optimization objective
modes_weight = 1.0
transfers_weight = 1.0

def calculate_objective_observation_based(df_evaluation):
    df_evaluation = pd.merge(df_reference, df_evaluation, on = "request_index", 
        suffixes = ["_reference", "_evaluation"])
    
    df_evaluation["offset"] = transfers_weight * np.abs(
        df_evaluation["transfers_reference"] - df_evaluation["transfers_evaluation"])

    for mode in modes:
        df_evaluation["offset"] += modes_weight * np.abs(
            df_evaluation["legs_{}_reference".format(mode)] - df_evaluation["legs_{}_evaluation".format(mode)]
        )

    return np.sum(df_evaluation["offset"] * df_evaluation["weight"]) / df_evaluation["weight"].sum()

def calculate_objective_distribution_based(df_evaluation):
    df_evaluation = pd.merge(df_reference, df_evaluation, on = "request_index", 
        suffixes = ["_reference", "_evaluation"])
    
    reference_mode_distribution = []
    evaluation_mode_distribution = []

    for mode in modes:
        reference_mode_distribution.append((df_evaluation["legs_{}_reference".format(mode)] * df_evaluation["weight"]).sum())
        evaluation_mode_distribution.append((df_evaluation["legs_{}_evaluation".format(mode)] * df_evaluation["weight"]).sum())

    reference_mode_distribution = np.array(reference_mode_distribution) / np.sum(reference_mode_distribution)
    evaluation_mode_distribution = np.array(evaluation_mode_distribution) / np.sum(evaluation_mode_distribution)

    reference_transfer_distribution = []
    evaluation_transfer_distribution = []

    for transfers in range(maximum_transfers + 1):
        f_reference = df_evaluation["transfers_reference"] == transfers
        reference_transfer_distribution.append(df_evaluation.loc[f_reference, "weight"].sum())

        f_evaluation = df_evaluation["transfers_evaluation"] == transfers
        evaluation_transfer_distribution.append(df_evaluation.loc[f_evaluation, "weight"].sum())

    reference_transfer_distribution = np.array(reference_transfer_distribution) / np.sum(reference_transfer_distribution)
    evaluation_transfer_distribution = np.array(evaluation_transfer_distribution) / np.sum(evaluation_transfer_distribution)

    mode_distribution_offset = np.abs(reference_mode_distribution - evaluation_mode_distribution)
    transfer_distribution_offset = np.abs(reference_transfer_distribution - evaluation_transfer_distribution)

    return transfers_weight * np.sum(transfer_distribution_offset) + modes_weight * np.sum(mode_distribution_offset)

In [27]:
# Prepare function to convert CMA-ES' candidate to utilities
def prepare_utilities(values):
    utilities = {}
    
    for variable in variables:
        if variable["index"] is not None:
            utilities[variable["name"]] = values[variable["index"]]
        else:
            utilities[variable["name"]] = variable["initial"]
    
    return utilities

In [28]:
# Prepare bounds and initial values
initial = []
bounds = [[], []]

for variable in variables:
    if "optimized" in variable:
        initial.append(variable["initial"])
        bounds[0].append(-np.inf)
        bounds[1].append(0.0)

In [29]:
# Test connection
assert len(query_endpoint(requests[:5], prepare_utilities(initial))) == 5

In [30]:
# Configure CMA-ES
seed = 1000
sigma = 0.5
iterations = 1000

options = cma.CMAOptions()
options.set("bounds", bounds)
options.set("seed", seed)

algorithm = cma.CMAEvolutionStrategy(initial, sigma, options)

# Load cached data for previous iterations
history = []

if os.path.exists(output_path):
    with open(output_path, "rb") as f:
        history = pickle.load(f)

        algorithm.feed_for_resume(
            [h["candidate"] for h in history[1:]], # first one is initial
            [h["objective"] for h in history[1:]]
        )

# Choose objective
if objective == "observation_based":
    calculate_objective = calculate_objective_observation_based
elif objective == "distribution_based":
    calculate_objective = calculate_objective_distribution_based
else:
    raise RuntimeError("Unknown objective")

# Perform a new batch of iterations
for iteration in range(iterations):
    initial_evaluation = len(history) == 0
    candidates = [initial]

    if not initial_evaluation:
        candidates = algorithm.ask()

    objectives = []

    for candidate in candidates:
        utilities = prepare_utilities(candidate)
        df_response = query_endpoint_batched(requests, utilities)
        objective = calculate_objective(df_response)

        objectives.append(objective)

        history.append({
            "candidate": candidate,
            "utilities": utilities,
            "objective": objective,
            "evaluation": df_response,
            "initial": initial_evaluation
        })

    if not initial_evaluation:
        algorithm.tell(candidates, objectives)
        algorithm.disp()

    # Save after a successful CMA-ES iteration
    with open(output_path, "wb+") as f:
        pickle.dump(history, f)

(4_w,9)-aCMA-ES (mu_w=2.8,w_1=49%) in dimension 6 (seed=1000, Thu Jan 23 10:38:26 2025)
Iterat #Fevals   function value  axis ratio  sigma  min&max std  t[m:s]
    1      9 4.893456657747085e+00 1.0e+00 5.04e-01  5e-01  5e-01 1:16.9
    2     18 5.051795750703385e+00 1.2e+00 5.68e-01  5e-01  7e-01 2:37.4
    3     27 5.226790464449977e+00 1.5e+00 5.79e-01  5e-01  7e-01 3:48.4
    4     36 5.172506504934496e+00 1.6e+00 5.63e-01  5e-01  7e-01 5:02.6
    5     45 5.079474168081119e+00 1.7e+00 6.14e-01  5e-01  7e-01 6:22.5
    6     54 4.600148155496024e+00 1.7e+00 5.59e-01  5e-01  7e-01 7:53.3
    7     63 4.632632105450663e+00 1.8e+00 5.60e-01  5e-01  7e-01 9:28.4
    8     72 4.753849671996791e+00 1.7e+00 6.51e-01  6e-01  8e-01 10:54.3
    9     81 4.652551330945272e+00 1.9e+00 7.47e-01  6e-01  9e-01 12:29.2
   10     90 4.611957511889679e+00 2.3e+00 7.26e-01  5e-01  9e-01 13:54.0
   11     99 4.682603437303536e+00 2.6e+00 6.89e-01  5e-01  9e-01 15:19.8
   12    108 4.536307539503931e+0

KeyboardInterrupt: 